<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
}
</style>

# Rotation Representations

The IMU measures along its own axes, and an arrow printed on the top side of the flight controller marks its forward direction. What generally matters, though, is the Duckiedrone's attitude (i.e., how it is turned) in the room, rather than the IMU itself.

The attitude of the Duckiedrone is defined by introducing two reference frames and the rotation between them.

The __body frame__ is fixed to the Duckiedrone, centered at its center of mass, and turns and moves along with it. To simplify the problem, the IMU axes are __assumed__ to coincide with the Duckiedrone's body axes (called the "good assembly" assumption below). In fact, on an assembled Duckiedrone the flight controller is positioned to be aligned with the Duckiedrone "front", as well as being more or less at its center of mass.

The __world frame__ is instead fixed to the room.

The orientation, or attitude, of the Duckiedrone is therefore the rotation that expresses the body frame in the world frame.

There are several ways to write this rotation down, and all of them describe the same physical orientation. This notebook covers two of them, Euler angles and the rotation matrix, and shows where Euler angles break down.

The widget at the end of this notebook uses an airplane instead of the Duckiedrone because its orientation is easier to see. The math is the same for the Duckiedrone.

By the end of the notebook you should be able to answer:

- What do the body frame and the world frame describe, and how do they define the attitude of the Duckiedrone?

- How do roll, pitch and yaw combine into one rotation in the ZYX convention?

- What is gimbal lock, and which motion does it take away?

## Euler angles and the rotation matrix

Both the body and world frames are Cartesian right-handed frames:

- the $x$ axis points "forward";

- the $z$ axis points "up" (opposite the direction of local gravitational acceleration);

- the $y$ axis points "left", in such a way to create a
  [right-handed reference frame](https://en.wikipedia.org/wiki/Right-hand_rule).

Given that only the orientation matters in this notebook (and not translations), both these frames are drawn with the same origin, and before any rotation they line up (i.e., when the attitude of the Duckiedrone is zero, however it is measured, the Duckiedrone is aligned perfectly with the world frame).

Call the body and world coordinate frames $\mathrm{B}$ and $\mathrm{W}$, respectively. For a fixed orientation, write $\mathbf{R}_{\mathrm{WB}}$ for the body-to-world rotation $\mathbf{R}_{\mathrm{WB}}(t)$ from [Notebook 1](./1-imu-basics.ipynb). Let $\mathbf{p}_{\mathrm{B}}$ and $\mathbf{p}_{\mathrm{W}}$ be the coordinates of the same point in those frames:

<span id="equation-1"></span>

$$
\mathbf{p}_{\mathrm{W}} = \mathbf{R}_{\mathrm{WB}}\,\mathbf{p}_{\mathrm{B}}.
\tag{1}
$$

A rotation matrix is dimensionless. A valid rotation matrix satisfies $\mathbf{R}_{\mathrm{WB}}^{\mathsf{T}}\mathbf{R}_{\mathrm{WB}} = \mathbf{I}$ and $\det\!\left(\mathbf{R}_{\mathrm{WB}}\right) = +1$. The first condition makes the columns unit length and mutually perpendicular. The second rules out a reflection, so the matrix preserves handedness.

One way to describe the orientation is by using three angles, each turning about one axis:

- roll $\alpha$, about $x$, the nose axis;

- pitch $\beta$, about $y$, the sideways axis;

- yaw $\gamma$, about $z$, the vertical axis.

They are often referred to as Euler angles. The angles are measured in $\mathrm{rad}$; $^\circ$ is used below where it makes an example easier to read.

[Figure 1](#figure-1) shows the roll, pitch, and yaw axes on the Duckiedrone.

<a id="figure-1"></a>

<p class="lx-figure" style="width:500px;">
  <img src="../assets/_images/duckiedrone_roll_pitch_yaw.jpg" style="width:100%;" />
</p>

Figure 1: Roll turns about the nose axis, pitch about the sideways axis, and yaw about the vertical axis.

### One rotation at a time

Write $p_x$, $p_y$, and $p_z$ for the components of $\mathbf{p}_{\mathrm{B}}$:

$$
\mathbf{p}_{\mathrm{B}} =
\begin{bmatrix}
p_x \\
p_y \\
p_z
\end{bmatrix}.
$$

Write $\mathbf{R}_i(\theta)$ for a right-handed rotation through angle $\theta$ about axis $i \in \{x, y, z\}$, as in the [notation table of Notebook 1](./1-imu-basics.ipynb#notation). A right-handed yaw by angle $\gamma$ has rotation matrix $\mathbf{R}_z(\gamma)$; applied to a point, it gives $\mathbf{p}_{\mathrm{W}} = \mathbf{R}_z(\gamma)\,\mathbf{p}_{\mathrm{B}}$:

<span id="equation-2"></span>

$$
\mathbf{R}_z(\gamma) =
\begin{bmatrix}
\cos(\gamma) & -\sin(\gamma) & 0 \\
\sin(\gamma) & \cos(\gamma) & 0 \\
0 & 0 & 1
\end{bmatrix}.
\tag{2}
$$

The $z$ coordinate is unchanged, because yaw turns about the $z$ axis, while $x$ and $y$ turn as a point on a circle does. The nose initially points along the body $x$-axis basis vector $\mathbf{e}_x$, with $\mathbf{e}_x = [1, 0, 0]^{\mathsf{T}}$. Take a Duckiedrone yawed by $90^\circ$. Therefore, $\cos(\gamma) = 0$ and $\sin(\gamma) = 1$. Substituting those values into [Equation (2)](#equation-2) gives:

<span id="equation-3"></span>

$$
\mathbf{R}_z(90^\circ)\,\mathbf{e}_x =
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
= \mathbf{e}_y.
\tag{3}
$$

The result is the world $y$-axis basis vector $\mathbf{e}_y$. Thus, the nose points along the world $y$ axis: a $\frac{1}{4}$ turn to the left. The pitch and roll rotation matrices each leave their own axis unchanged. The pitch matrix $\mathbf{R}_y(\beta)$ is:

$$
\mathbf{R}_y(\beta) =
\begin{bmatrix}
\cos(\beta) & 0 & \sin(\beta) \\
0 & 1 & 0 \\
-\sin(\beta) & 0 & \cos(\beta)
\end{bmatrix}.
$$

With $y$ pointing left, the right-hand rule makes a positive pitch tip the nose down.

The roll matrix $\mathbf{R}_x(\alpha)$ is:

$$
\mathbf{R}_x(\alpha) =
\begin{bmatrix}
1 & 0 & 0 \\
0 & \cos(\alpha) & -\sin(\alpha) \\
0 & \sin(\alpha) & \cos(\alpha)
\end{bmatrix}.
$$

### Putting the three together

Any arbitrary rotation in 3D space can be decomposed into three subsequent rotations:

<span id="equation-4"></span>

$$
\mathbf{R}_{\mathrm{WB}} = \mathbf{R}_z(\gamma)\,\mathbf{R}_y(\beta)\,\mathbf{R}_x(\alpha).
\tag{4}
$$

Read about the fixed world axes, the rightmost matrix acts first: roll, then pitch, then yaw. Read about the Duckiedrone's own axes, the same product is yaw, then pitch, then roll. [Equation (4)](#equation-4) defines the ZYX convention used in this notebook: it maps body-frame components into world-frame components. Other sources can use different orders, directions, or frame conventions, so the formula must always accompany the name.

The order matters, because matrix multiplication does not generally commute. Rolling by $90^\circ$ and then pitching by $90^\circ$, both about the fixed world axes, leaves the nose pointing down; pitching first and then rolling leaves it pointing left. Therefore, the order is part of the convention and has to be stated.

### The rotation matrix is a representation too

$\mathbf{R}_{\mathrm{WB}}$ is not only a step on the way to Euler angles: its nine numbers are themselves a way to write an orientation, and they have a direct meaning. Its first column is $\mathbf{R}_{\mathrm{WB}}\,\mathbf{e}_x$, the direction the body $x$ axis points in the world, and the second and third columns are the body $y$ and $z$ axes. For the yawed Duckiedrone above, the first column of $\mathbf{R}_z(90^\circ)$ is $\mathbf{e}_y$, the nose direction derived in [Equation (3)](#equation-3).

Under the good assembly assumption, the IMU reports vectors in the body frame, and [Equation (1)](#equation-1) maps their components into the world frame. The price is nine numbers for something three angles already describe, and each column must keep length 1 and stay at right angles to the others to preserve pure rotation (without dilations).

## Gimbal lock

[Figure 2](#figure-2) shows the two angles used to locate a point on a sphere.

<a id="figure-2"></a>

<p class="lx-figure" style="width:250px;">
  <img src="../assets/_images/3d_unit_sphere.jpg" style="width:100%;" />
</p>

Figure 2: Two angles name every point on a sphere: the longitude $\theta$, measured around the equator, and the latitude $\phi$, measured up from the equator. At the poles, $\theta$ is undefined.

Consider a point on a sphere; two angles are sufficient to identify it: longitude and latitude, as on a globe. The trouble is at the north pole, where the latitude is $90^\circ$ and every longitude gives the same point. Someone walking straight over the pole never changes direction, yet their longitude jumps by $180^\circ$ the moment they cross. Nothing is wrong with the sphere or the person's movement; it is the mathematical notation describing the angles that somehow breaks.

Euler angles break in the same way when pitch reaches $\beta = \pm 90^\circ$, read "plus or minus $90^\circ$." At $\beta = -90^\circ$ the nose of the Duckiedrone points straight up. Roll spins it about its nose, which now points up, and yaw spins it about the vertical axis. Roll and yaw now do the same thing, so only their sum $\alpha + \gamma$ matters.

[Figure 3](#figure-3) contrasts level flight with the pitch angle where roll and yaw overlap.

<a id="figure-3"></a>

<p class="lx-figure" style="width:760px;">
  <img src="../assets/_images/duckiedrone_pitch_0.jpg" style="width:48%; display:inline-block;" />
  <img src="../assets/_images/duckiedrone_pitch_90.jpg" style="width:48%; display:inline-block;" />
</p>

Figure 3: Left: the Duckiedrone level, nose along $x$. Right: pitched to $\beta = -90^\circ$, nose pointing up. Roll and yaw now turn it about the same vertical line, so the blue marker on its $+y$ side can only move around a circle.

With two angles doing the same job, one kind of motion is left without an angle to produce it. Pitch still tips the nose toward $x$, but no small change of the three angles tips it sideways, toward $y$. To tilt the nose sideways by just $0.001\,\mathrm{rad}$, yaw has to jump by $90^\circ$, and roll by $90^\circ$ the other way to keep the rest of the orientation in place: the pole crossing again. That is gimbal lock: at one orientation, one direction of rotation is lost.

The name comes from the gimbal, a set of three nested rings that each turn about one axis. Turn the middle ring by $90^\circ$ and the other two line up, so the device physically cannot turn about the third axis. The video [Euler (gimbal lock) Explained](https://youtu.be/zc8b2Jo7mno?t=39) animates this with a set of gimbal rings. The activity below shows the same effect on a toy airplane.

## Activity 1: watch gimbal lock happen

<a id="activity-1"></a>

The widget below rotates a toy airplane through [Equation (4)](#equation-4). The red dot is the nose, and the blue dot is the wing tip on the $+y$ side, like the blue marker in [Figure 3](#figure-3).

With pitch at zero, each slider moves the airplane in a different way. Now set pitch to $-90^\circ$ and, before touching the other two sliders, predict: can roll and yaw each be changed by $30^\circ$ in a way that leaves the airplane exactly where it is? Then try it. The readout above the plot reports the only combination of roll and yaw that still matters at this orientation.

Repeat at pitch $+90^\circ$, where the nose points down. The readout now reports roll minus yaw: find which way roll and yaw have to change together for the airplane to stay where it is.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import FloatSlider, interact
from mpl_toolkits.mplot3d import Axes3D

# Symbols follow the notebook:
#   roll, pitch, yaw   The angles alpha, beta, and gamma, in degrees for the sliders.
#   a, b, g            The same three angles in radians, as used in the matrices.

# A toy airplane: nose first, then fuselage, tail fin, and the two wings.
airplane = np.array(
    (
        np.concatenate(
            (
                1.15,
                np.linspace(1, -0.75, 25),
                np.linspace(-0.75, -1, 15),
                np.linspace(0.4, 0, 10),
                np.linspace(0.4, 0, 10),
            ),
            axis=None,
        ),
        np.concatenate(
            (
                0,
                np.linspace(0, 0, 25),
                np.linspace(0, 0, 15),
                np.linspace(0, 0.9, 10),
                np.linspace(0, -0.9, 10),
            ),
            axis=None,
        ),
        np.concatenate(
            (
                0,
                np.linspace(0, 0, 25),
                np.linspace(0, 0.5, 15),
                np.linspace(0, 0, 10),
                np.linspace(0, 0, 10),
            ),
            axis=None,
        ),
    )
)
WING_TIP = np.argmax(airplane[1])  # The tip of the wing on the +y side.


def rotation_matrix(roll_deg, pitch_deg, yaw_deg):
    """Build R = RZ @ RY @ RX, the ZYX convention from the section above."""
    a, b, g = np.deg2rad([roll_deg, pitch_deg, yaw_deg])
    rx = np.array(
        [[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]]
    )
    ry = np.array(
        [[np.cos(b), 0, np.sin(b)], [0, 1, 0], [-np.sin(b), 0, np.cos(b)]]
    )
    rz = np.array(
        [[np.cos(g), -np.sin(g), 0], [np.sin(g), np.cos(g), 0], [0, 0, 1]]
    )
    return rz @ ry @ rx


def show(roll=0.0, pitch=0.0, yaw=0.0):
    pts = rotation_matrix(roll, pitch, yaw) @ airplane

    fig = plt.figure(figsize=(6.5, 6))
    ax = fig.add_subplot(projection=Axes3D.name)
    ax.view_init(elev=10, azim=30)

    for vec in [(1.5, 0, 0), (0, 1.5, 0), (0, 0, 1.5)]:
        ax.quiver(0, 0, 0, *vec, color="0.6", arrow_length_ratio=0.06)
    ax.text(1.3, 0, 0, "x", color="0.4")
    ax.text(0, 1.1, 0, "y", color="0.4")
    ax.text(0, 0, 1.1, "z", color="0.4")

    ax.scatter(pts[0, 1:], pts[1, 1:], pts[2, 1:], c="black", s=60)
    ax.scatter(pts[0, 0], pts[1, 0], pts[2, 0], c="red", s=120)
    ax.scatter(*pts[:, WING_TIP], c="tab:blue", s=120)

    for lim in (ax.set_xlim3d, ax.set_ylim3d, ax.set_zlim3d):
        lim(-1.2, 1.2)

    if abs(abs(pitch) - 90.0) < 1.0:
        # Nose up: the two angles add; nose down: they subtract.
        op, locked = ("+", roll + yaw) if pitch < 0 else ("-", roll - yaw)
        ax.set_title(
            f"gimbal lock: roll and yaw act on the same axis\n"
            f"only roll {op} yaw matters, and it is {locked:+.0f} deg",
            color="tab:red",
        )
    else:
        ax.set_title(
            f"roll {roll:+.0f} deg   pitch {pitch:+.0f} deg   yaw {yaw:+.0f} deg"
        )

    fig.tight_layout()
    plt.show()


interact(
    show,
    roll=FloatSlider(
        value=0.0,
        min=-180,
        max=180,
        step=1,
        description="roll",
        continuous_update=False,
    ),
    pitch=FloatSlider(
        value=0.0,
        min=-90,
        max=90,
        step=1,
        description="pitch",
        continuous_update=False,
    ),
    yaw=FloatSlider(
        value=0.0,
        min=-180,
        max=180,
        step=1,
        description="yaw",
        continuous_update=False,
    ),
);

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
}
</style>

Gimbal lock belongs to the Euler angles, not to the Duckiedrone. The rotation matrix describes the nose-up orientation without any trouble: its first column is simply $\mathbf{e}_z$. A hovering Duckiedrone never pitches near $\pm 90^\circ$, and [Notebook 4](./4-quaternions-and-rotation-representations.ipynb) starts from why gimbal lock still matters for it.

## Further reading

3Blue1Brown's [Three-dimensional linear transformations](https://www.youtube.com/watch?v=rHLEWRxRGiM) shows visually why the columns are where the axes land. The derivation of the three matrices is in the Duckietown [Representations - Part 1](https://hubs.ly/Q02bk6-V0) slides and the [associated lecture video](https://vimeo.com/588472832?fl=pl&fe=sh), and Chapter 3 of the free textbook [Modern Robotics](https://hades.mech.northwestern.edu/index.php/Modern_Robotics) covers rotation matrices in full.

[Gimbal lock](https://en.wikipedia.org/wiki/Gimbal_lock) explains how two rotation axes can coincide.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()